# Weather Data Processing with Polars (2015–2025)

This notebook processes daily weather observations collected from NOAA weather stations in Poland using Polars.

Main objectives:
- load and process prepared weather data efficiently,
- filter observations for Polish weather stations,
- calculate temperature and weather metric aggregates,
- compare data availability across stations and metrics,
- prepare analysis-ready summary tables for further exploration.

In [ ]:
# Import libraries
import os  # Used to read environment variables with os.getenv().
from dotenv import load_dotenv  # Used to load variables from the .env file.
from sqlalchemy import create_engine
from pathlib import Path
import polars as pl

## Database configuration for sample analysis

This notebook uses the Docker PostgreSQL database with sample weather data.

The sample database is used to validate the Bronze → Silver → Gold pipeline and to demonstrate how the analysis can read from the Gold layer.

The full NOAA dataset is not loaded into Docker by default.

In [2]:
# # Define project paths and load local database credentials from the .env file
# PROJECT_PATH = Path(
#     r"C:\Users\zychl\Documents\GitHub\data-toolkit\weather_data_processing"
# )

# PLOTS_PATH = PROJECT_PATH / "02_plots"

# load_dotenv(PROJECT_PATH / ".env") 

# # PostgreSQL connection details
# username = os.getenv('POSTGRES_USER')
# password = os.getenv('POSTGRES_PASSWORD')
# host = 'localhost'
# port = os.getenv('HOST_POSTGRES_PORT')
# database = os.getenv('POSTGRES_DB')

# engine = create_engine(
#     f'postgresql+pg8000://{username}:{password}@{host}:{port}/{database}'
#     )

# # Fetch the data from PostgreSQL analytics_db server - gold layer.
# # Open a temporary SQLAlchemy connection for this Polars query.
# with engine.connect() as conn:
#     df_raw = pl.read_database(
#         query='SELECT * FROM gold.weather_observations',
#         connection=conn
#     )

## Database configuration for full analysis

This notebook uses the local full PostgreSQL database for analysis, not the Docker demo database.

The Docker database contains only sample data and is used to validate the Bronze → Silver → Gold pipeline.
The local analysis database contains the full prepared dataset used for the 2015–2025 analysis.

In [3]:
PROJECT_PATH = Path(
    r'C:\Users\zychl\Documents\GitHub\data-toolkit\weather_data_processing'
)

PLOTS_PATH = PROJECT_PATH / "02_plots"

# Load database configuration for the local full analysis database.
# This is intentionally separate from the Docker .env file, because Docker uses
# sample data for pipeline validation, while this notebook analyzes the local
# full prepared dataset
load_dotenv(PROJECT_PATH / ".env.analysis") 

username = os.getenv('ANALYSIS_DB_USER')
password = os.getenv('ANALYSIS_DB_PASSWORD')
host = 'localhost'
port = os.getenv('ANALYSIS_DB_PORT')
database = os.getenv('ANALYSIS_DB_NAME')

engine = create_engine(
    f'postgresql+pg8000://{username}:{password}@{host}:{port}/{database}'
    )

with engine.connect() as conn:
    df_raw = pl.read_database(
        query='SELECT * FROM gold.weather_observations',
        connection=conn
    )

### Inspect df

In [4]:
# Check min and max dates
df_raw.select([
    pl.col("observation_date").min().alias("min_date"),
    pl.col("observation_date").max().alias("max_date")
])

min_date,max_date
date,date
2015-01-01,2026-06-29


In [5]:
# Preview the DataFrame
df_raw.head()

station,station_name,elevation,observation_date,month_year,metric,value
str,str,"decimal[38,1]",date,str,str,"decimal[38,1]"
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""TMAX""",5.4
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""TMIN""",1.2
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""PRCP""",0.0
"""PL000012120""","""Leba""",2.0,2015-01-01,"""01-2015""","""TAVG""",4.0
"""PL000012385""","""Siedlce""",152.0,2015-01-01,"""01-2015""","""TMAX""",1.5


In [6]:
# Check DataFrame shape
df_raw.shape

(137418, 7)

In [7]:
# Check column names and data types
df_raw.schema

Schema([('station', String),
        ('station_name', String),
        ('elevation', Decimal(precision=38, scale=1)),
        ('observation_date', Date),
        ('month_year', String),
        ('metric', String),
        ('value', Decimal(precision=38, scale=1))])

In [8]:
# Check memory used by DataFrame
df_raw.estimated_size("mb")

8.647311210632324

In [9]:
# Count null values in each column
df_raw.null_count()

station,station_name,elevation,observation_date,month_year,metric,value
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0


In [10]:
# Count duplicated rows
df_raw.is_duplicated().sum()

0

In [11]:
# Count unique values in each column
df_raw.select(pl.all().n_unique())

station,station_name,elevation,observation_date,month_year,metric,value
u32,u32,u32,u32,u32,u32,u32
10,10,10,4175,138,5,785


In [12]:
df_raw.select([
    pl.col('station_name').unique().alias('unique_stations')
])


unique_stations
str
"""Okecie"""
"""Siedlce"""
"""Strachowice"""
"""Szczecin"""
"""Bialystok"""
"""Lawica"""
"""Wlodawa"""
"""Balice"""
"""Leba"""


## Analyze the trend in Poland's average annual temperature (2015-2025)

#### Extract TAVG metric and calculate min/max date

In [13]:
# Filter to TAVG observations before 2026 and keep only columns needed for analysis.
from datetime import date

df = df_raw.filter([
    (pl.col('metric') == 'TAVG') &
    (pl.col('observation_date') < date(2026, 1, 1))  
]).sort('observation_date').select(
    pl.col([
        'station', 'station_name', 'elevation', 'observation_date', 'metric', 'value'
]))

df.head()

station,station_name,elevation,observation_date,metric,value
str,str,"decimal[38,1]",date,str,"decimal[38,1]"
"""PL000012120""","""Leba""",2.0,2015-01-01,"""TAVG""",4.0
"""PL000012385""","""Siedlce""",152.0,2015-01-01,"""TAVG""",0.7
"""PLM00012160""","""Elblag-Milejewo""",43.0,2015-01-01,"""TAVG""",1.4
"""PLM00012205""","""Szczecin""",7.0,2015-01-01,"""TAVG""",3.0
"""PLM00012295""","""Bialystok""",151.0,2015-01-01,"""TAVG""",0.9


In [14]:
# Verify the date range after filtering to TAVG observations from 2015–2025.
df.select([
    pl.col('observation_date').min().alias('min_date'),
    pl.col('observation_date').max().alias('max_date')
])

min_date,max_date
date,date
2015-01-01,2025-12-31


In [15]:
# Add year extracted from observation_date.
df = df.with_columns(
    year = pl.col('observation_date').dt.year()
)

df.head()

station,station_name,elevation,observation_date,metric,value,year
str,str,"decimal[38,1]",date,str,"decimal[38,1]",i32
"""PL000012120""","""Leba""",2.0,2015-01-01,"""TAVG""",4.0,2015
"""PL000012385""","""Siedlce""",152.0,2015-01-01,"""TAVG""",0.7,2015
"""PLM00012160""","""Elblag-Milejewo""",43.0,2015-01-01,"""TAVG""",1.4,2015
"""PLM00012205""","""Szczecin""",7.0,2015-01-01,"""TAVG""",3.0,2015
"""PLM00012295""","""Bialystok""",151.0,2015-01-01,"""TAVG""",0.9,2015


#### Data quality assessment

To improve the reliability of the analysis, station coverage was evaluated before calculating annual average temperatures.<br>
Stations with sufficient temporal coverage across the study period were selected to minimize bias resulting from changes in station availability over time.

In [16]:
# Count the number of observations for each weather station
# to evaluate station coverage and data completeness.
dq_count = df.group_by('station_name').agg(
    pl.col('observation_date').count().alias('observation_count')
    ).sort(
        'observation_count', 
        descending=True
        )

dq_count

station_name,observation_count
str,u32
"""Szczecin""",4000
"""Siedlce""",3999
"""Leba""",3999
"""Wlodawa""",3999
"""Bialystok""",3999
"""Elblag-Milejewo""",3998
"""Balice""",3876
"""Okecie""",3876
"""Strachowice""",3876


The full 2015–2025 period contains 4018 calendar days, including leap years 2016, 2020 and 2024. Station coverage close to 4018 observations indicates nearly complete daily TAVG coverage.

##### Further investigate Balice, Ławica, Okęcie and Strachowice

In [17]:
# Define stations to include
include = [
    'Balice',
    'Lawica',
    'Okecie',
    'Strachowice'
]

# Filter the dataset to selected stations
stations4 = df.filter(pl.col('station_name').is_in(include))

# Check whether the stations were filtered correctly
stations4.select(
    pl.col('station_name').unique()
    )

station_name
str
"""Okecie"""
"""Balice"""
"""Lawica"""
"""Strachowice"""


In [18]:
stations4.schema

Schema([('station', String),
        ('station_name', String),
        ('elevation', Decimal(precision=38, scale=1)),
        ('observation_date', Date),
        ('metric', String),
        ('value', Decimal(precision=38, scale=1)),
        ('year', Int32)])

In [19]:
# Drop unnecessary columns
remove_cols = [
    'station',
    'elevation', 
    'metric', 
    'value',
]

stations_table = stations4.drop(remove_cols)
stations_table

# Aggregate date ranges and observation counts by station
agg_stations = stations_table.group_by('station_name').agg(
    pl.col('observation_date').min().alias('min_date'),
    pl.col('observation_date').max().alias('max_date'),
    pl.col('observation_date').count().alias('observation_count')
    )

agg_stations

station_name,min_date,max_date,observation_count
str,date,date,u32
"""Balice""",2015-01-01,2025-08-24,3876
"""Okecie""",2015-01-01,2025-08-24,3876
"""Strachowice""",2015-01-01,2025-08-24,3876
"""Lawica""",2015-01-01,2025-08-24,3876


In [20]:
agg_all = df.group_by('station_name').agg(
    pl.col('observation_date').min().alias('min_date'),
    pl.col('observation_date').max().alias('max_date'),
    pl.col('observation_date').count().alias('observation_count')
    ).sort(
        pl.col('observation_count'),
        descending=True
    )

agg_all

station_name,min_date,max_date,observation_count
str,date,date,u32
"""Szczecin""",2015-01-01,2025-12-31,4000
"""Siedlce""",2015-01-01,2025-12-31,3999
"""Wlodawa""",2015-01-01,2025-12-31,3999
"""Leba""",2015-01-01,2025-12-31,3999
"""Bialystok""",2015-01-01,2025-12-31,3999
"""Elblag-Milejewo""",2015-01-01,2025-12-31,3998
"""Strachowice""",2015-01-01,2025-08-24,3876
"""Balice""",2015-01-01,2025-08-24,3876
"""Okecie""",2015-01-01,2025-08-24,3876


#### Summary
The selected stations cover the entire analysis period (2015–2025). Four stations contain slightly fewer TAVG observations than the others, indicating the presence of missing daily records. However, all stations span the full observation period, suggesting that the missing observations are distributed within the time series rather than caused by a shorter reporting period.

### Annual average temperature in Poland (2015-2025)

In [21]:
# Group by year
avg_temp_agg = df.group_by('year').agg(
    pl.col('value').mean().alias('avg_temp')
    ).sort(['year', 'avg_temp'])

avg_temp_agg

year,avg_temp
i32,f64
2015,9.793041
2016,9.134836
2017,8.934329
2018,9.814458
2019,10.193918
…,…
2021,8.74595
2022,9.486466
2023,10.047918


### Annual Average Temperature by Station (2015–2025)

In [22]:
# Group by year and station_name
station_avg_temp = df.group_by(['year', 'station_name']).agg(
    pl.col('value').mean().alias('avg_temp')
    ).sort(
        ['year', 'avg_temp'],
        descending=[False, True]
    )

station_avg_temp

year,station_name,avg_temp
i32,str,f64
2015,"""Strachowice""",11.621644
2015,"""Lawica""",10.635068
2015,"""Okecie""",10.504658
2015,"""Balice""",10.01589
2015,"""Szczecin""",9.781644
…,…,…
2025,"""Wlodawa""",9.425833
2025,"""Leba""",9.408056
2025,"""Siedlce""",9.345833


### Correlation Between Elevation and Average Temperature (2015-2025)

In [34]:
# Verify that the dataset contains only the average temperature metric
df.select(pl.col('metric').unique())

metric
str
"""TAVG"""


In [31]:
elev_corr_prep = df.group_by(['station_name', 'elevation']).agg(
    pl.col('value').mean().alias('avg_temp')
    ).sort(
        ['elevation', 'avg_temp'],
        descending=[True, True]
    )

elev_corr = elev_corr_prep[['elevation', 'avg_temp']]
elev_corr.corr()

elevation,avg_temp
f64,f64
1.0,0.100411
0.100411,1.0


The Pearson correlation coefficient (r = 0.10) indicates a very weak positive relationship between station elevation and average temperature. The wide dispersion of data points around the regression line suggests that elevation alone does not explain temperature differences among the analyzed weather stations. Other geographical and climatic factors are likely to have a much greater influence.

### Seasonal Average Temperature by Weather Station

In [ ]:
# Select columns needed for the seasonal temperature analysis
heatmap_raw = df[('station_name', 'observation_date', 'metric', 'value')]
heatmap_raw

heatmap = heatmap_raw.clone()
heatmap

station_name,observation_date,metric,value
str,date,str,"decimal[38,1]"
"""Leba""",2015-01-01,"""TAVG""",4.0
"""Siedlce""",2015-01-01,"""TAVG""",0.7
"""Elblag-Milejewo""",2015-01-01,"""TAVG""",1.4
"""Szczecin""",2015-01-01,"""TAVG""",3.0
"""Bialystok""",2015-01-01,"""TAVG""",0.9
…,…,…,…
"""Siedlce""",2025-12-31,"""TAVG""",-4.1
"""Elblag-Milejewo""",2025-12-31,"""TAVG""",-4.0
"""Szczecin""",2025-12-31,"""TAVG""",-1.1


In [69]:
# 1. Extract corresponding month number.
heatmap = heatmap.with_columns(
    month = pl.col('observation_date').dt.month()
)

# 2. Use extracted month number to assign seasons:
#       12, 1, 2 → Winter
#       3, 4, 5 → Spring
#       6, 7, 8 → Summer
#       9, 10, 11 → Autumn

heatmap = heatmap.with_columns(
    # Use pl.lit() to treat season names as literal string values instead of column names
    pl.when(pl.col('month').is_in([12,1,2])).then(pl.lit('Winter'))
      .when(pl.col('month').is_in([3,4,5])).then(pl.lit('Spring'))
      .when(pl.col('month').is_in([6,7,8])).then(pl.lit('Summer'))
      .when(pl.col('month').is_in([9,10,11])).then(pl.lit('Autumn'))
      .otherwise(pl.lit('Other'))
      .alias('season')
)

# 3. # Calculate average seasonal temperature for each station across the full analysis period
heatmap_agg = heatmap.group_by(['station_name', 'season']).agg(
    pl.col('value').mean().alias('avg_temp')
)

# 4. Pivot long to wide
heatmap_matrix = heatmap_agg.pivot(
    on='season',
    index='station_name',
    values='avg_temp'
)

# 5. Change the order of Seasons.
heatmap_matrix = heatmap_matrix[
    [
        'station_name',
        'Winter',
        'Spring',
        'Summer',
        'Autumn'
    ]
]

heatmap_matrix

station_name,Winter,Spring,Summer,Autumn
str,f64,f64,f64,f64
"""Lawica""",1.810428,9.513307,19.703685,10.292715
"""Balice""",0.904797,9.242701,19.62251,9.839294
"""Elblag-Milejewo""",0.190293,7.525323,17.441724,9.022155
"""Siedlce""",0.439737,8.636743,18.984936,9.236419
"""Strachowice""",2.342961,9.722145,19.886056,10.691943
"""Okecie""",1.241606,9.714796,20.188446,10.169095
"""Szczecin""",2.208898,8.896723,18.469673,10.172864
"""Bialystok""",-0.231345,8.050348,18.410902,8.610463
"""Leba""",2.132356,7.613505,17.485233,10.32827
